# Master Notebook Utilities
Shared utility functions for master notebook execution

In [ ]:
# Imports needed by utility functions
from notebookutils import mssparkutils
import json
import ast
import re
import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import current_timestamp, lit

In [ ]:
# File and String Utilities
def get_file_content_using_notebookutils(file):
    data = spark.sparkContext.wholeTextFiles(file).collect()
    file_content = data[0][1]
    return file_content

def parse_exit_value(exit_val_str):
    """Parse exit value string and return timing dict with datetime objects, or None if unparseable."""
    if not exit_val_str:
        return None
    
    try:
        parsed = ast.literal_eval(exit_val_str)
        
        # Convert ISO timestamp strings to datetime objects
        start_time = parsed.get('start_time')
        end_time = parsed.get('end_time')
        
        if start_time and isinstance(start_time, str):
            start_time = datetime.fromisoformat(start_time)
        
        if end_time and isinstance(end_time, str):
            end_time = datetime.fromisoformat(end_time)
        
        return {
            'start_time': start_time,
            'end_time': end_time,
            'execution_time': parsed.get('execution_time')
        }
    except (ValueError, SyntaxError):
        return None

def get_hash(file, hashes):
    """Get hash for a specific file from hashes list."""
    for h in hashes:
        if h['file'] == file:
            return h['hash']
    return ""

In [ ]:
# DAG Building Functions
def build_dag_from_manifest(manifest, cell_timeout=90, select_str="", exclude_str=""):
    """
    Build DAG activities from manifest for execution with runMultiple.
    
    Args:
        manifest: Parsed dbt manifest dictionary
        cell_timeout: Timeout per cell in seconds (default: 90)
        select_str: Optional selection string (e.g., "model1 +model2 tag:daily")
        exclude_str: Optional exclusion string (e.g., "model3 tag:deprecated")
    
    Returns:
        List of activity dictionaries ready for runMultiple
    """
    # If selection/exclusion provided, resolve which nodes to include
    if select_str or exclude_str:
        selected_node_ids = resolve_model_selection(manifest, select_str, exclude_str)
        # Filter manifest nodes to only selected ones
        all_nodes = {
            node_id: node 
            for node_id, node in manifest.get('nodes', {}).items()
            if node_id in selected_node_ids
        }
    else:
        # No selection = all executable nodes (models, snapshots, seeds, tests)
        all_nodes = {
            node_id: node 
            for node_id, node in manifest.get('nodes', {}).items()
            if node.get('resource_type') in ['model', 'snapshot', 'seed', 'test']
        }
    
    if len(all_nodes) == 0:
        raise Exception("No models found matching selection criteria. Check your select/exclude syntax.")
    
    # Build DAG activities structure
    dag_activities = []
    
    for node_id, node in all_nodes.items():
        activity = {
            'name': node_id,  # unique_id (e.g., "model.my_project.my_model")
            'path': node_id,  # notebook name matches unique_id
            'timeoutPerCellInSeconds': cell_timeout,
            'args': {}
        }
        
        # Add dependencies (only those that are also in the selected set)
        deps = node.get('depends_on', {}).get('nodes', [])
        filtered_deps = [dep for dep in deps if dep in all_nodes]
        if filtered_deps:
            activity['dependencies'] = filtered_deps
        
        dag_activities.append(activity)
    
    return dag_activities

In [ ]:
# Model Selection Resolution
def resolve_model_selection(manifest, select_str, exclude_str):
    """
    Resolve dbt selection syntax to a set of node IDs.
    
    Supports:
    - Direct names: "my_model"
    - Graph operators: "+my_model" (upstream), "my_model+" (downstream), "+my_model+" (both)
    - Tags: "tag:daily"
    - Paths: "path:models/staging"  
    - Wildcards: "stg_*"
    - Multiple selectors: space-separated
    
    Args:
        manifest: Parsed dbt manifest dictionary
        select_str: Selection string (space-separated selectors)
        exclude_str: Exclusion string (space-separated selectors)
    
    Returns:
        Set of node IDs to execute (includes resolved dependencies)
    """
    # Helper: Build reverse dependency map for downstream resolution
    def build_reverse_deps(manifest):
        reverse_deps = {}
        for node_id, node in manifest.get('nodes', {}).items():
            for dep in node.get('depends_on', {}).get('nodes', []):
                if dep not in reverse_deps:
                    reverse_deps[dep] = []
                reverse_deps[dep].append(node_id)
        return reverse_deps
    
    # Helper: Get all upstream dependencies recursively
    def get_upstream_deps(node_id, manifest_nodes, visited=None):
        if visited is None:
            visited = set()
        if node_id in visited or node_id not in manifest_nodes:
            return visited
        visited.add(node_id)
        for dep in manifest_nodes[node_id].get('depends_on', {}).get('nodes', []):
            get_upstream_deps(dep, manifest_nodes, visited)
        return visited
    
    # Helper: Get all downstream dependencies recursively
    def get_downstream_deps(node_id, reverse_deps, visited=None):
        if visited is None:
            visited = set()
        if node_id in visited:
            return visited
        visited.add(node_id)
        for dep in reverse_deps.get(node_id, []):
            get_downstream_deps(dep, reverse_deps, visited)
        return visited
    
    # Helper: Match a single selector against manifest
    def match_selector(selector, manifest_nodes, reverse_deps):
        matched = set()
        
        # Parse graph operators
        upstream = selector.startswith('+')
        downstream = selector.endswith('+')
        selector_clean = selector.strip('+')
        
        # Parse selector type
        if selector_clean.startswith('tag:'):
            # Tag selector
            tag = selector_clean[4:]
            for node_id, node in manifest_nodes.items():
                if tag in node.get('tags', []):
                    matched.add(node_id)
        
        elif selector_clean.startswith('path:'):
            # Path selector
            path = selector_clean[5:]
            for node_id, node in manifest_nodes.items():
                if node.get('original_file_path', '').startswith(path):
                    matched.add(node_id)
        
        elif '*' in selector_clean:
            # Wildcard selector (simple glob matching)
            pattern = re.escape(selector_clean).replace(r'\*', '.*')
            regex = re.compile(pattern)
            for node_id, node in manifest_nodes.items():
                node_name = node.get('name', '')
                if regex.match(node_name):
                    matched.add(node_id)
        
        else:
            # Direct name match (unique_id or name)
            for node_id, node in manifest_nodes.items():
                if selector_clean == node.get('name') or selector_clean == node_id:
                    matched.add(node_id)
        
        # Apply graph operators
        result = set()
        for node_id in matched:
            if upstream and not downstream:
                # +model (upstream + self)
                result.update(get_upstream_deps(node_id, manifest_nodes))
            elif downstream and not upstream:
                # model+ (self + downstream)
                result.add(node_id)
                result.update(get_downstream_deps(node_id, reverse_deps))
            elif upstream and downstream:
                # +model+ (upstream + self + downstream)
                result.update(get_upstream_deps(node_id, manifest_nodes))
                result.update(get_downstream_deps(node_id, reverse_deps))
            else:
                # model (just self, no dependencies)
                result.add(node_id)
        
        return result
    
    # Main logic
    manifest_nodes = manifest.get('nodes', {})
    reverse_deps = build_reverse_deps(manifest)
    selected = set()
    excluded = set()
    
    # Process selections
    if select_str:
        for selector in select_str.split():
            selected.update(match_selector(selector, manifest_nodes, reverse_deps))
    else:
        # No selection = all executable nodes
        selected = {
            node_id for node_id in manifest_nodes.keys()
            if manifest_nodes[node_id].get('resource_type') in ['model', 'snapshot', 'seed', 'test']
        }
    
    # Process exclusions
    if exclude_str:
        for selector in exclude_str.split():
            excluded.update(match_selector(selector, manifest_nodes, reverse_deps))
    
    # Apply exclusions
    final = selected - excluded
    
    # Only return executable nodes (models, snapshots, seeds, tests - filter out sources, etc.)
    final_nodes = {
        node_id for node_id in final
        if manifest_nodes.get(node_id, {}).get('resource_type') in ['model', 'snapshot', 'seed', 'test']
    }
    
    return final_nodes

In [ ]:
# Logging and Database Utilities
def ensure_column_exists(table_name, column_name, column_type):
    """Add column to table if it doesn't exist. Uses SQL for DDL operation."""
    schema_check_sql = f"DESCRIBE {table_name}"
    schema_check_df = spark.sql(schema_check_sql)
    
    if column_name not in [row['col_name'] for row in schema_check_df.collect()]:
        alter_table_sql = f"ALTER TABLE {table_name} ADD COLUMN {column_name} {column_type}"
        spark.sql(alter_table_sql)

def insert_new_batch(batch_id, master_notebook, log_lakehouse):
    """Insert new batch record using DataFrame API."""
    from pyspark.sql import Row
    
    batch_record = Row(
        batch_id=batch_id,
        start_time=datetime.now(),
        status='open',
        master_notebook=master_notebook
    )
    
    df = spark.createDataFrame([batch_record])
    df.writeTo(f"{log_lakehouse}.batch").append()

def close_batch(batch_id, master_notebook, status, log_lakehouse):
    """Close batch by updating status using Delta merge."""
    from delta.tables import DeltaTable
    from pyspark.sql import Row
    
    # Create update DataFrame
    update_record = Row(batch_id=batch_id, master_notebook=master_notebook, status=status)
    update_df = spark.createDataFrame([update_record])
    
    # Get Delta table
    batch_table = DeltaTable.forName(spark, f"{log_lakehouse}.batch")
    
    # Merge to update status
    batch_table.alias("target").merge(
        update_df.alias("source"),
        "target.batch_id = source.batch_id AND target.master_notebook = source.master_notebook"
    ).whenMatchedUpdate(
        set={"status": "source.status"}
    ).execute()

In [ ]:
# Result Processing Functions
def format_failure_summary(failed_notebooks, preview_length):
    """Format failure summary for error reporting."""
    error_lines = [f"\nFailed notebooks ({len(failed_notebooks)}):\n"]
    
    for failure in failed_notebooks:
        notebook_name = failure['notebook']
        error_msg = failure['error']
        error_preview = error_msg[:preview_length] + "..." if len(error_msg) > preview_length else error_msg
        
        error_lines.append(f"{notebook_name}")
        error_lines.append(f"  └─ {error_preview}")
        error_lines.append("")
    
    return "\n".join(error_lines)

def verify_metadata_hashes(embedded_hashes, embedded_hashcheck, metadata_path):
    """Verify metadata hashes match between embedded and environment."""
    if embedded_hashcheck == 0:
        print('Metadata Hash Check Bypassed')
        return
    
    current_hashes = json.loads(get_file_content_using_notebookutils(metadata_path + 'MetaHashes.json'))
    
    if current_hashes != embedded_hashes:
        for h in embedded_hashes:
            print(
                h['file'] + '\n \t Emb Hash: ' + get_hash(h['file'], embedded_hashes) + 
                '\n \t Env Hash: ' + get_hash(h['file'], current_hashes)
            )
        
        if embedded_hashcheck == 1:
            print('Warning!: Hashes do not match. Its recommended to re-generate the dbt project using the latest extract of the target environment metadata.')
        else:
            raise Exception('ERROR, Hashes do not match. Its recommended to re-generate the dbt project using the latest extract of the target environment metadata.')
    else:
        print('Metadata Hashes Match 😏')

In [ ]:
# High-Level Orchestration Functions

def setup_logging(log_lakehouse):
    """
    Setup all logging infrastructure: create tables, ensure schema, start new batch.
    
    Args:
        log_lakehouse: Name of the lakehouse for logging
    
    Returns:
        tuple: (batch_id, master_notebook_name)
    """
    # Create execution_log table
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {log_lakehouse}.execution_log (
          notebook STRING,
          start_time TIMESTAMP,
          end_time TIMESTAMP,
          status STRING,
          error STRING,
          batch_id STRING,
          master_notebook STRING,
          execution_time DOUBLE
        )
        USING DELTA
    ''')
    
    # Create batch table
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {log_lakehouse}.batch (
          batch_id STRING,
          start_time TIMESTAMP,
          status STRING,
          master_notebook STRING
        )
        USING DELTA
    ''')
    
    # Ensure required columns exist (for schema evolution)
    ensure_column_exists(f'{log_lakehouse}.execution_log', 'master_notebook', 'STRING')
    ensure_column_exists(f'{log_lakehouse}.batch', 'master_notebook', 'STRING')
    
    # Start new batch
    batch_id = str(uuid.uuid4())
    master_notebook = mssparkutils.runtime.context.get('currentNotebookName')
    insert_new_batch(batch_id, master_notebook, log_lakehouse)
    
    print(f"Batch ID: {batch_id}")
    return batch_id, master_notebook


def execute_dbt_dag(manifest_path, select, exclude, cell_timeout, dag_timeout, concurrency):
    """
    Execute dbt DAG with runMultiple, handling errors gracefully.
    
    Args:
        manifest_path: Base path to metadata directory (manifest.json will be appended)
        select: Selection criteria (empty string for all)
        exclude: Exclusion criteria (empty string for none)
        cell_timeout: Timeout per cell in seconds
        dag_timeout: Total DAG timeout in seconds
        concurrency: Number of parallel workers
    
    Returns:
        dict: Result dictionary from runMultiple
    """
    # Load manifest
    manifest_json = get_file_content_using_notebookutils(manifest_path + "manifest.json")
    manifest = json.loads(manifest_json)
    
    # Build DAG activities
    dag_activities = build_dag_from_manifest(
        manifest, 
        cell_timeout=cell_timeout, 
        select_str=select, 
        exclude_str=exclude
    )
    
    print(f"Found {len(dag_activities)} notebook(s)")
    if select:
        print(f"Selection: {select}")
    if exclude:
        print(f"Exclusion: {exclude}")
    print()
    print(f"Concurrency: {concurrency} threads")
    print()
    
    if len(dag_activities) == 0:
        raise Exception("No models to execute. Check your selection syntax or dbt project configuration.")
    
    # Build DAG configuration
    DAG = {
        "activities": dag_activities,
        "timeoutInSeconds": dag_timeout,
        "concurrency": concurrency
    }
    
    # Execute with error handling
    try:
        result = mssparkutils.notebook.runMultiple(DAG, {"displayDAGViaGraphviz": False})
    except Exception as e:
        if "RunMultipleFailedException" in str(type(e)):
            result = e.result
        else:
            raise
    
    return result


def log_execution_results(result, batch_id, master_notebook, log_lakehouse, execution_start_time):
    """
    Log execution results to Delta tables and close batch.
    
    Args:
        result: Result dictionary from runMultiple
        batch_id: Batch identifier
        master_notebook: Name of master notebook
        log_lakehouse: Name of logging lakehouse
        execution_start_time: Timestamp when master execution started
    
    """
    execution_records = []
    failed_count = 0
    
    schema = StructType([
        StructField("notebook", StringType(), False),
        StructField("start_time", TimestampType(), False),  # Non-nullable
        StructField("end_time", TimestampType(), False),    # Non-nullable
        StructField("status", StringType(), False),
        StructField("error", StringType(), True),
        StructField("batch_id", StringType(), False),
        StructField("master_notebook", StringType(), False),
        StructField("execution_time", DoubleType(), False)  # Non-nullable
    ])
    
    for activity_name, activity_result in result.items():
        has_error = activity_result.get('exception') is not None
        timing = parse_exit_value(activity_result.get('exitVal'))
        
        # Handle missing timing use execution start time for fallback
        if not timing:
            start_time = execution_start_time
            end_time = datetime.now()
            execution_time = 0.0  # Indicates no timing available
        else:
            # Use actual timing data
            start_time = timing['start_time']
            end_time = timing['end_time']
            execution_time = timing['execution_time']
        
        if has_error:
            failed_count += 1
        
        error_msg = str(activity_result.get('exception')) if has_error else None
        status = 'error' if has_error else 'success'
        
        execution_records.append((
            activity_name,
            start_time,
            end_time,
            status,
            error_msg,
            batch_id,
            master_notebook,
            execution_time
        ))
    
    # Write to execution log
    if execution_records:
        df = spark.createDataFrame(execution_records, schema)
        df.writeTo(f"{log_lakehouse}.execution_log").append()
    
    # Close batch
    batch_status = 'failed' if failed_count > 0 else 'closed'
    close_batch(batch_id, master_notebook, batch_status, log_lakehouse)


def print_execution_summary(result, error_preview_length):
    """
    Print human-readable execution summary to console.

    Args:
        result: Result dictionary from runMultiple
        error_preview_length: Max length for error message preview
    """
    failed_notebooks = []
    success_count = 0

    for activity_name, activity_result in result.items():
        has_error = activity_result.get('exception') is not None

        if has_error:
            error_msg = str(activity_result.get('exception'))
            # Extract just the error type (e.g., DELTA_MERGE_UNRESOLVED_EXPRESSION)
            error_type = ""
            if '[' in error_msg and ']' in error_msg:
                error_type = error_msg[error_msg.find('[')+1:error_msg.find(']')]
            elif 'failed due to upstream' in error_msg.lower():
                error_type = "upstream failure"
            else:
                error_type = error_msg[:80]

            failed_notebooks.append({
                'notebook': activity_name,
                'error_type': error_type
            })
        else:
            success_count += 1

    total = len(result)
    failed_count = len(failed_notebooks)

    print()
    print(f"Completed successfully: {success_count}/{total}")

    if failed_count > 0:
        print(f"Errors: {failed_count}")
        print()
        for failure in failed_notebooks:
            notebook_name = failure['notebook']
            error_type = failure['error_type']
            print(f"  {notebook_name} - {error_type}")

    print()
    print(f"Done. PASS={success_count} WARN=0 ERROR={failed_count} TOTAL={total}")